# VAW Layer 3 - Colab run: automatic temporal labels (VLM pseudo-labels)
Needs a GPU runtime (T4). Needs notebooks 01-02 (clean clips) done. For every violent clip a vision-language model is asked twice (different frame sampling and wording) when violence begins and when the lead-up begins; only clips where both answers agree become training labels.
These are PSEUDO-labels: their accuracy is measured later against the human validation labels (step 3). Resumable per clip (one JSON each).

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r requirements.txt bitsandbytes accelerate
!python -m pytest -q

## 1. Run the proposals (prints `vlm: n/N clips`; a cell prints its summary only at the end)
If the T4 runs out of memory, use the 4B model: add `--model Qwen/Qwen3-VL-4B-Instruct`.

In [ ]:
!python -m src.label.vlm_propose --config configs/colab.yaml

## 2. Look at a few answers
Each JSON holds both answers and whether they agreed.

In [ ]:
import glob, json
for f in sorted(glob.glob('data/label/vlm/*/*.json'))[:8]:
    d = json.load(open(f)); print(d['clip_id'], d['status'], d['answers'])

## 3. After the human validation labels exist (`data/label/human_labels.csv`, from the labeling page) measure how accurate the pseudo-labels are
This is the number that decides whether the automatic route can be trusted.

In [ ]:
!python -m src.label.vlm_propose --config configs/colab.yaml --score data/label/human_labels.csv